# OmniBird-xattn on CIFAR-10 (40% sparse pool)

The full **xattn** recipe ported to the original PointBigBird CIFAR-10 setup. Why we expect this to work where the event-camera xattn struggled: each pixel token here carries `3 × 8 = 24 bits` of RGB content vs. `1 bit` of polarity for events. Token entropy is the load-bearing variable — the same JEPA recipe is stable on data-rich tokens.

**The recipe (all our hard-won fixes):**

| Piece | What it does |
|---|---|
| BigBird per-event encoder | Sparse attention with multi-curve serialization. Now consuming 2D tokens `(y, x, R, G, B)`. |
| `FixedPosEmbedder` shared | NeRF γ → frozen orthogonal projection, single shared instance everywhere. |
| `LocalCrossAttention` pool | `scores -= α · ‖q_coord − k_coord‖²` — spatial-locality bias forces local attention. |
| **Tokenizer skip-connection** at pool input | `pool_kv = encoder(events) + tokenizer(s, c)` — guarantees per-event variation regardless of encoder collapse. |
| **Symmetric cross-attended pool** on context and target | Predictor's job is "pool-at-target ← pool-at-context", same statistic at different locations (true i-JEPA contract). |
| **Target centering before LayerNorm** (DINO-style EMA of batch mean) | The actual anti-collapse mechanism — kills the "all targets point in one direction" minimum that pure per-token LN cannot. |
| Multi-block masking at patch level | 4 target blocks × `patches_per_block` patches. Standard i-JEPA. |
| Smooth-L1 loss on `LN(centered(target))` | No VICReg, no other regularizer. |
| EMA momentum ramp 0.999 → 0.9999 | Standard. |


## 1. Setup

In [ ]:
import os, sys, math, time, copy, ssl
sys.path.insert(0, os.path.abspath('..'))
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import torchvision

from omnibird import (
    OmniBirdConfig,
    BigBirdEventEncoderWithPool, PerceiverPredictor, FixedPosEmbedder,
    TargetCenter, ema_update, make_momentum_schedule,
    jepa_loss, diag_dict, fmt_diag,
    save_atomic, ensure_dir, short_params,
    precompute_grid_orderings, invert_perm, quantize_coords,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)

N_GPUS_REQUESTED = 4
N_GPUS = min(N_GPUS_REQUESTED, torch.cuda.device_count())
USE_DP = (DEVICE == "cuda") and (N_GPUS > 1)
print(f"GPUs visible = {torch.cuda.device_count()}  using = {N_GPUS}  DataParallel = {USE_DP}")

cfg = OmniBirdConfig()
cfg.coord_dim       = 2                # (y, x)
cfg.signal_dim      = 3                # (R, G, B)
cfg.side            = 32               # 32x32 CIFAR-10 grid
cfg.image_size      = 32
cfg.frac_pool       = 0.4              # 40% of pixels per image
cfg.n_events_total  = 0
cfg.n_events_max    = 400              # K_pool = 400 pixels per image

# Patch layout
cfg.patch_mode         = True
cfg.patch_size         = 10            # 10 pixels per patch
cfg.patches_per_block  = 4
cfg.n_pred_blocks      = 4
cfg.n_tgt              = cfg.n_pred_blocks * cfg.patches_per_block      # 16
cfg.ctx_max_patches    = 16
cfg.patch_curve        = "hilbert"

# Position embedding
cfg.pos_embed       = "nerf"
cfg.nerf_freqs      = 5

# Encoder / predictor sizes
cfg.d_model          = 256
cfg.n_layers_enc     = 6
cfg.n_heads          = 8
cfg.dim_head         = 32
cfg.d_pred           = 256             # = d_model for shared fixed pos embedder
cfg.n_layers_pred    = 4
cfg.n_heads_pred     = 6
cfg.dim_head_pred    = 32
cfg.ffn_mult         = 4
cfg.fourier_dim      = 96
cfg.fourier_scale    = 15.0
cfg.predictor_pos_symmetric = True

# BigBird sparse attention
cfg.attention_type   = "bigbird"
cfg.block_size       = 8
cfg.window           = 1
cfg.n_random         = 2
cfg.n_global         = 2
cfg.reinject_pos     = False

# Pool locality bias (LocalCrossAttention)
cfg.local_bias_scale = 10.0

# JEPA: target centering + per-token LN + smooth_L1
cfg.loss_type        = "smooth_l1"
cfg.use_centering    = True
cfg.ema_start        = 0.999
cfg.ema_end          = 0.9999

# Optim
cfg.epochs         = 200
cfg.batch_size     = 128
cfg.lr             = 1e-3
cfg.weight_decay   = 0.05
cfg.warmup_epochs  = 10
cfg.probe_interval = 10
cfg.probe_epochs   = 2
cfg.log_every      = 50
cfg.ckpt_dir       = "./checkpoints_cifar10_xattn"
ensure_dir(cfg.ckpt_dir)

P_max = cfg.n_events_max // cfg.patch_size
print(f"K_pool = {cfg.n_events_max}  patch_size = {cfg.patch_size}  P_max = {P_max}")
print(f"context = {cfg.ctx_max_patches} patches  target = {cfg.n_tgt} patches")


## 2. CIFAR-10 sparse-pool patch dataset

For each image:
1. Sample 40% of pixels (400 of 1024) with a per-image fixed RNG.
2. Sort pixel indices by 2-D Hilbert curve so contiguous chunks are spatially coherent.
3. Reshape into `P=40` patches of `K=10` pixels each.
4. i-JEPA-style multi-block masking on the patches: 4 target blocks × 4 patches, ~16 context patches from the remaining.

Returns the same dict format `OmniBirdPatchDataset` returns so the encoder pipeline plugs in unchanged.


In [ ]:
# Precompute Hilbert ranks for the 32x32 grid (2D).
GRID_RANKS = {k: v for k, v in precompute_grid_orderings(cfg.image_size, ndim=2).items()}

class CIFAR10SparsePatchDataset(Dataset):
    def __init__(self, base, train=True,
                 image_size=cfg.image_size, frac_pool=cfg.frac_pool,
                 K_pool=cfg.n_events_max, patch_size=cfg.patch_size,
                 patches_per_block=cfg.patches_per_block,
                 n_pred_blocks=cfg.n_pred_blocks,
                 ctx_max=cfg.ctx_max_patches,
                 curve="hilbert", pool_seed=0):
        self.base = base
        self.train = train
        self.image_size = image_size
        self.K_pool  = K_pool
        self.K_per   = patch_size
        self.patches_per_block = patches_per_block
        self.n_pred_blocks     = n_pred_blocks
        self.ctx_max = ctx_max
        self.P_max = K_pool // patch_size
        self.curve = curve

        # Per-image fixed pool index permutation (so the SAME 40% pixels are
        # used every epoch for the same image, but different images see
        # different subsets).
        N_pix = image_size * image_size
        rng = np.random.RandomState(pool_seed)
        self.pool_idx = np.stack(
            [rng.permutation(N_pix)[:K_pool] for _ in range(len(base))],
            axis=0,
        ).astype(np.int64)

        # Full grid coords in [-1, 1]
        ys, xs = torch.meshgrid(
            torch.linspace(-1.0, 1.0, image_size),
            torch.linspace(-1.0, 1.0, image_size),
            indexing='ij',
        )
        self.coords_all = torch.stack([ys, xs], dim=-1).view(N_pix, 2).float()

    def __len__(self): return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]                       # img: (3, H, W) in [0, 1]
        if isinstance(img, torch.Tensor):
            img_t = img.float()
        else:
            img_t = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        # Normalize to [-1, 1]
        img_t = img_t * 2.0 - 1.0
        rgb = img_t.permute(1, 2, 0).reshape(-1, 3)        # (N_pix, 3)

        # Gather pool pixels (fixed per idx)
        pool_idx = self.pool_idx[idx]                       # (K_pool,)
        pool_coords = self.coords_all[pool_idx]             # (K_pool, 2) in [-1, 1]
        pool_signal = rgb[pool_idx]                         # (K_pool, 3) in [-1, 1]

        # Sort by Hilbert curve (over the full grid then index by pool members)
        full_ranks = GRID_RANKS[self.curve]                 # (N_pix,) Long
        pool_ranks = full_ranks[pool_idx]
        order = pool_ranks.argsort()
        pool_coords = pool_coords[order]
        pool_signal = pool_signal[order]

        # Reshape to (P, K, coord_dim + signal_dim)
        P = self.P_max; K = self.K_per
        ev_combined = torch.cat([pool_coords, pool_signal], dim=-1)  # (K_pool, 5)
        patch_events  = ev_combined.view(P, K, 5)
        patch_centroids = pool_coords.view(P, K, 2).mean(dim=1)     # (P, 2)
        # All real (no padding in the CIFAR-10 case — K_pool=400 fills P*K exactly)
        patch_kpm = torch.zeros(P, dtype=torch.bool)
        event_kpm_per_patch = torch.zeros(P, K, dtype=torch.bool)

        if self.train:
            rng = np.random.RandomState()
            forbidden = np.zeros(P, dtype=bool)
            # Sample target blocks
            tgt_blocks = []
            for _ in range(self.n_pred_blocks):
                allowed = np.where(~forbidden)[0]
                if len(allowed) < self.patches_per_block:
                    break
                a = allowed[rng.randint(len(allowed))]
                d2 = ((patch_centroids - patch_centroids[a]) ** 2).sum(-1).numpy()
                d2[forbidden] = np.inf
                blk = np.argsort(d2, kind="stable")[:self.patches_per_block]
                tgt_blocks.append(blk)
                forbidden[blk] = True
            tgt_idx = np.concatenate(tgt_blocks).astype(np.int64) if tgt_blocks \
                       else np.zeros(0, dtype=np.int64)
            expected_tgt = self.n_pred_blocks * self.patches_per_block
            if len(tgt_idx) < expected_tgt:
                fill_val = int(tgt_idx[-1]) if len(tgt_idx) else 0
                fill = np.full(expected_tgt - len(tgt_idx), fill_val, dtype=np.int64)
                tgt_idx = np.concatenate([tgt_idx, fill]) if len(tgt_idx) else fill

            allowed = np.where(~forbidden)[0]
            if len(allowed) > self.ctx_max:
                ctx_idx = allowed[rng.choice(len(allowed), self.ctx_max, replace=False)]
                ctx_idx.sort()
            else:
                ctx_idx = allowed
            if len(ctx_idx) < self.ctx_max:
                fill_val = int(ctx_idx[0]) if len(ctx_idx) else 0
                fill = np.full(self.ctx_max - len(ctx_idx), fill_val, dtype=ctx_idx.dtype)
                ctx_idx = np.concatenate([ctx_idx, fill])

            tgt_patch_centroids = patch_centroids[torch.from_numpy(tgt_idx)]
            sample = {
                "pool_patch_events":    patch_events.contiguous(),
                "pool_patch_centroids": patch_centroids.contiguous(),
                "pool_patch_kpm":       patch_kpm.contiguous(),
                "pool_event_kpm":       event_kpm_per_patch.contiguous(),
                "ctx_patch_idx":        torch.from_numpy(ctx_idx).contiguous(),
                "tgt_patch_idx":        torch.from_numpy(tgt_idx).contiguous(),
                "tgt_patch_centroids":  tgt_patch_centroids.contiguous(),
                "label":                int(label),
            }
            return {k: v.clone() if isinstance(v, torch.Tensor) else v for k, v in sample.items()}

        # Test path: use all real patches as ctx, no targets
        real_idx = torch.arange(P)
        ctx_idx = real_idx[:self.ctx_max] if len(real_idx) >= self.ctx_max else \
                  torch.cat([real_idx, torch.zeros(self.ctx_max - len(real_idx), dtype=torch.long)])
        sample = {
            "pool_patch_events":    patch_events.contiguous(),
            "pool_patch_centroids": patch_centroids.contiguous(),
            "pool_patch_kpm":       patch_kpm.contiguous(),
            "pool_event_kpm":       event_kpm_per_patch.contiguous(),
            "ctx_patch_idx":        ctx_idx.contiguous(),
            "label":                int(label),
        }
        return {k: v.clone() if isinstance(v, torch.Tensor) else v for k, v in sample.items()}


CIFAR_ROOT = os.path.expanduser("~/data/cifar10")
os.makedirs(CIFAR_ROOT, exist_ok=True)

cifar_train = torchvision.datasets.CIFAR10(root=CIFAR_ROOT, train=True,  download=True)
cifar_test  = torchvision.datasets.CIFAR10(root=CIFAR_ROOT, train=False, download=True)

train_ds      = CIFAR10SparsePatchDataset(cifar_train, train=True,  pool_seed=0)
train_eval_ds = CIFAR10SparsePatchDataset(cifar_train, train=False, pool_seed=0)
test_ds       = CIFAR10SparsePatchDataset(cifar_test,  train=False, pool_seed=1)

train_loader      = DataLoader(train_ds,      batch_size=cfg.batch_size, shuffle=True,  num_workers=0)
train_eval_loader = DataLoader(train_eval_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=0)
test_loader       = DataLoader(test_ds,       batch_size=cfg.batch_size, shuffle=False, num_workers=0)
print(f"train={len(train_ds)}  test={len(test_ds)}  pool_K={cfg.n_events_max}  P={train_ds.P_max}")


## 3. Visualize masking

In [ ]:
classes = ['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']
sample = train_ds[0]
P, K, Fdim = sample["pool_patch_events"].shape
pool_ev = sample["pool_patch_events"].numpy()      # (P, K, 5) — first 2 dims = coords
ctx_idx = sample["ctx_patch_idx"].numpy()
tgt_idx = sample["tgt_patch_idx"].numpy()
print(f"label = {classes[sample['label']]}, P={P}, K={K}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# (a) Hilbert-sorted patches colored by index
ax = axes[0]
rng_cmap = np.random.RandomState(0)
patch_colors = plt.cm.tab20(rng_cmap.permutation(P) % 20)
for p in range(P):
    coords = pool_ev[p, :, :2]
    ax.scatter(coords[:, 1], -coords[:, 0], s=15, c=[patch_colors[p]] * K)
ax.set_aspect('equal'); ax.set_title(f"(a) Hilbert-sorted patches"); ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05)

# (b) Target blocks
TGT_COLORS = ['#fbbf24', '#34d399', '#60a5fa', '#f472b6']
ax = axes[1]
for p in range(P):
    coords = pool_ev[p, :, :2]
    ax.scatter(coords[:, 1], -coords[:, 0], s=10, c='lightgray', alpha=0.5)
for k in range(cfg.n_pred_blocks):
    block_ids = tgt_idx[k*cfg.patches_per_block:(k+1)*cfg.patches_per_block]
    for p in block_ids:
        coords = pool_ev[p, :, :2]
        ax.scatter(coords[:, 1], -coords[:, 0], s=20, c=TGT_COLORS[k], label=f'block {k+1}' if p == block_ids[0] else None)
ax.set_aspect('equal'); ax.set_title(f"(b) 4 target blocks × {cfg.patches_per_block} patches"); ax.legend(fontsize=8)
ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05)

# (c) Context + targets overlay
ax = axes[2]
for p in range(P):
    coords = pool_ev[p, :, :2]
    in_ctx = p in ctx_idx; in_tgt = p in tgt_idx
    if in_ctx:
        ax.scatter(coords[:, 1], -coords[:, 0], s=15, c='#ef4444', label='ctx' if p == ctx_idx[0] else None)
    elif in_tgt:
        k = np.argmax([p in tgt_idx[i*cfg.patches_per_block:(i+1)*cfg.patches_per_block] for i in range(cfg.n_pred_blocks)])
        ax.scatter(coords[:, 1], -coords[:, 0], s=15, c=TGT_COLORS[k])
    else:
        ax.scatter(coords[:, 1], -coords[:, 0], s=8, c='lightgray', alpha=0.4)
ax.set_aspect('equal'); ax.set_title(f"(c) red=context, colors=targets, gray=leftover")
ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05)
plt.tight_layout(); plt.show()


## 4. Models — symmetric xattn pool + tokenizer skip + target centering

In [ ]:
# Shared fixed positional embedder (orthogonal-projected NeRF, frozen)
shared_pos = FixedPosEmbedder(coord_dim=cfg.coord_dim, d_model=cfg.d_model,
                               num_freqs=cfg.nerf_freqs).to(DEVICE)

def make_encoder():
    return BigBirdEventEncoderWithPool(
        d_model=cfg.d_model, n_layers=cfg.n_layers_enc,
        n_heads=cfg.n_heads, dim_head=cfg.dim_head,
        block_size=cfg.block_size, window=cfg.window,
        n_random=cfg.n_random, n_global=cfg.n_global,
        ffn_mult=cfg.ffn_mult,
        signal_dim=cfg.signal_dim, coord_dim=cfg.coord_dim,
        fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
        serial_orders=("z", "z_rev", "hilbert", "hilbert_rev"),
        reinject_pos=cfg.reinject_pos,
        side=cfg.side,
        fixed_pos_embedder=shared_pos,
        local_bias_scale=cfg.local_bias_scale,
    )

context_encoder = make_encoder().to(DEVICE)
target_encoder  = copy.deepcopy(context_encoder).to(DEVICE)
for q in target_encoder.parameters(): q.requires_grad_(False)

predictor = PerceiverPredictor(
    d_model=cfg.d_model, d_pred=cfg.d_pred,
    n_layers=cfg.n_layers_pred,
    n_heads=cfg.n_heads_pred, dim_head=cfg.dim_head_pred,
    coord_dim=cfg.coord_dim,
    fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
    ffn_mult=cfg.ffn_mult, pos_symmetric=cfg.predictor_pos_symmetric,
    pos_embed=cfg.pos_embed, nerf_freqs=cfg.nerf_freqs,
    fixed_pos_embedder=shared_pos,
).to(DEVICE)
target_center = TargetCenter(cfg.d_model, momentum=0.9).to(DEVICE)

if USE_DP:
    device_ids = list(range(N_GPUS))
    context_encoder = nn.DataParallel(context_encoder, device_ids=device_ids)
    target_encoder  = nn.DataParallel(target_encoder,  device_ids=device_ids)
    predictor       = nn.DataParallel(predictor,       device_ids=device_ids)

def _unwrap(m): return m.module if isinstance(m, nn.DataParallel) else m
print(f"context_encoder: {short_params(_unwrap(context_encoder))}")
print(f"predictor      : {short_params(_unwrap(predictor))}")


## 5. Optim + resume

In [ ]:
params = list(_unwrap(context_encoder).parameters()) + list(_unwrap(predictor).parameters())
optimizer = AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
total_steps = cfg.epochs * len(train_loader)
warmup_steps = cfg.warmup_epochs * len(train_loader)
def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    p = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1 + math.cos(math.pi * p))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
momentum_gen = make_momentum_schedule(cfg.ema_start, cfg.ema_end, total_steps)

LAST = os.path.join(cfg.ckpt_dir, "cifar10_xattn_last.pt")
BEST = os.path.join(cfg.ckpt_dir, "cifar10_xattn_best.pt")
RESUME = True
history = {"loss": [], "diag_steps": [], "diag_log": [], "probe_accs": []}
start_epoch, best_loss, global_step = 1, float("inf"), 0
m = cfg.ema_start
if RESUME and os.path.exists(LAST):
    s = torch.load(LAST, map_location=DEVICE, weights_only=False)
    _unwrap(context_encoder).load_state_dict(s["context_encoder"])
    _unwrap(target_encoder).load_state_dict(s["target_encoder"])
    _unwrap(predictor).load_state_dict(s["predictor"])
    target_center.load_state_dict(s["center"])
    optimizer.load_state_dict(s["opt"]); scheduler.load_state_dict(s["sched"])
    history = s.get("history", history)
    global_step = s.get("global_step", 0); best_loss = s.get("best_loss", float("inf"))
    start_epoch = s["epoch"] + 1
    for _ in range(global_step):
        try: m = next(momentum_gen)
        except StopIteration: m = cfg.ema_end
    print(f"resumed @ ep {s['epoch']}, step {global_step}")
else:
    print("starting fresh.")


## 6. Helpers + probe

In [ ]:
def gather_ctx_subset(batch):
    pool_ev    = batch["pool_patch_events"].to(DEVICE)
    pool_cen   = batch["pool_patch_centroids"].to(DEVICE)
    pool_evkpm = batch["pool_event_kpm"].to(DEVICE)
    ctx_idx    = batch["ctx_patch_idx"].to(DEVICE)
    B, P, K, Fd = pool_ev.shape
    Pc = ctx_idx.size(1)
    sub_ev    = torch.gather(pool_ev,    1, ctx_idx[..., None, None].expand(B, Pc, K, Fd))
    sub_cen   = torch.gather(pool_cen,   1, ctx_idx[..., None].expand(B, Pc, pool_cen.size(-1)))
    sub_evkpm = torch.gather(pool_evkpm, 1, ctx_idx[..., None].expand(B, Pc, K))
    return sub_ev.reshape(B, Pc*K, Fd), sub_cen, sub_evkpm.reshape(B, Pc*K)


def flatten_pool(batch):
    pool_ev    = batch["pool_patch_events"].to(DEVICE)
    pool_cen   = batch["pool_patch_centroids"].to(DEVICE)
    pool_evkpm = batch["pool_event_kpm"].to(DEVICE)
    B, P, K, Fd = pool_ev.shape
    return pool_ev.view(B, P*K, Fd), pool_cen, pool_evkpm.view(B, P*K)


class LinearProbe(nn.Module):
    def __init__(self, d, n=10):
        super().__init__()
        self.fc = nn.Linear(d, n)
    def forward(self, z): return self.fc(z)


def quick_probe(num_epochs=cfg.probe_epochs, lr=1e-3, wd=1e-4):
    enc = _unwrap(context_encoder)
    saved_rg = [p.requires_grad for p in enc.parameters()]
    for p in enc.parameters(): p.requires_grad_(False)
    enc.eval()
    probe = LinearProbe(cfg.d_model, 10).to(DEVICE)
    opt = AdamW(probe.parameters(), lr=lr, weight_decay=wd)
    ce  = nn.CrossEntropyLoss()
    def _z(b):
        ev, cen, kpm = gather_ctx_subset(b)
        with torch.no_grad():
            g = enc(ev, cen, event_kpm=kpm)              # (B, Pc, D)
        return g.mean(dim=1)
    for _ in range(num_epochs):
        probe.train()
        for b in train_eval_loader:
            y = b["label"].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            ce(probe(_z(b)), y).backward()
            opt.step()
    probe.eval()
    correct = total = 0
    with torch.no_grad():
        for b in test_loader:
            y = b["label"].to(DEVICE)
            correct += (probe(_z(b)).argmax(-1) == y).sum().item(); total += y.size(0)
    for p, rg in zip(enc.parameters(), saved_rg): p.requires_grad_(rg)
    return correct / max(total, 1)


## 7. Training loop

In [ ]:
epoch = start_epoch - 1
try:
    for epoch in range(start_epoch, cfg.epochs + 1):
        context_encoder.train(); predictor.train()
        epoch_loss, steps = 0.0, 0
        t0 = time.time()

        for batch in train_loader:
            tgt_cen = batch["tgt_patch_centroids"].to(DEVICE)

            # Target: full pool → encoder + LocalCrossAttention pool with target
            # centroids as queries. Center → LN.
            with torch.no_grad():
                pool_ev, _, pool_kpm = flatten_pool(batch)
                h_tgt_raw = target_encoder(pool_ev, tgt_cen, event_kpm=pool_kpm)
                Dim = h_tgt_raw.size(-1)
                if cfg.use_centering:
                    target_center.update(h_tgt_raw)
                    h_tgt = F.layer_norm(target_center(h_tgt_raw), (Dim,))
                else:
                    h_tgt = F.layer_norm(h_tgt_raw, (Dim,))

            # Context
            ctx_ev, ctx_cen, ctx_kpm = gather_ctx_subset(batch)
            g_ctx = context_encoder(ctx_ev, ctx_cen, event_kpm=ctx_kpm)

            # Predictor
            h_pred = predictor(g_ctx, tgt_cen,
                                ctx_coords=ctx_cen if cfg.predictor_pos_symmetric else None,
                                ctx_key_padding_mask=None)

            loss = jepa_loss(h_pred, h_tgt, loss_type=cfg.loss_type)
            optimizer.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step()

            try: m = next(momentum_gen)
            except StopIteration: m = cfg.ema_end
            ema_update(_unwrap(target_encoder), _unwrap(context_encoder), m)

            global_step += 1; epoch_loss += loss.item(); steps += 1
            if global_step % cfg.log_every == 0:
                pred_std = h_pred.std(dim=0).mean().item()
                tgt_std  = h_tgt.std(dim=0).mean().item()
                cos = F.cosine_similarity(h_pred, h_tgt, dim=-1).mean().item()
                print(f"[ep{epoch:02d} st{global_step:05d}]  loss={loss.item():.4f}  "
                      f"pred_std={pred_std:.3f}  tgt_std={tgt_std:.3f}  cos={cos:.3f}  "
                      f"lr={scheduler.get_last_lr()[0]:.1e}  m={m:.5f}")
                history["diag_steps"].append(global_step)
                history["diag_log"].append({"pred_std": pred_std, "tgt_std": tgt_std,
                                              "cos": cos, "loss": loss.item()})

        avg = epoch_loss / max(steps, 1)
        history["loss"].append(avg)
        improved = avg < best_loss
        if improved: best_loss = avg
        state = {
            "epoch": epoch, "cfg": cfg.__dict__,
            "context_encoder": _unwrap(context_encoder).state_dict(),
            "target_encoder":  _unwrap(target_encoder).state_dict(),
            "predictor":       _unwrap(predictor).state_dict(),
            "center":          target_center.state_dict(),
            "opt": optimizer.state_dict(), "sched": scheduler.state_dict(),
            "global_step": global_step, "best_loss": best_loss, "history": history,
        }
        save_atomic(state, LAST)
        if improved: save_atomic(state, BEST)
        print(f"=== ep {epoch:03d}/{cfg.epochs}  loss={avg:.4f}  m={m:.5f}  "
              f"{time.time()-t0:.1f}s" + ("  *" if improved else ""))

        if cfg.probe_interval > 0 and epoch % cfg.probe_interval == 0:
            t_p = time.time()
            acc = quick_probe()
            history["probe_accs"].append((epoch, acc))
            print(f"  [probe ep {epoch:03d}]  test_acc = {acc:.4f}  ({time.time()-t_p:.1f}s)")

    print("\nDone.")
except KeyboardInterrupt:
    print(f"\nInterrupted at epoch {epoch}.  Checkpoints in {cfg.ckpt_dir}.")


## 8. Final linear probe — standalone, run AFTER training

Loads the best checkpoint, freezes the context encoder, trains a linear classifier on mean-pooled context features for `n_probe_epochs=30`, evaluates on the held-out test split. This is the canonical "linear probe" protocol (longer + more careful than the 2-epoch quick-probe used every 10 epochs during training).

In [ ]:
LOAD_BEST = True
n_probe_epochs = 30
probe_lr       = 1e-3
probe_wd       = 1e-4

if LOAD_BEST and os.path.exists(BEST):
    print(f"loading best checkpoint @ {BEST}")
    s = torch.load(BEST, map_location=DEVICE, weights_only=False)
    _unwrap(context_encoder).load_state_dict(s["context_encoder"])
    print(f"  ckpt epoch = {s['epoch']}, train_loss = {s.get('best_loss', 'n/a')}")
else:
    print("using current in-memory weights")

enc = _unwrap(context_encoder)
enc.eval()
for p in enc.parameters():
    p.requires_grad_(False)

class FinalLinearProbe(nn.Module):
    def __init__(self, d, n=10):
        super().__init__()
        self.fc = nn.Linear(d, n)
    def forward(self, z): return self.fc(z)

probe = FinalLinearProbe(cfg.d_model, 10).to(DEVICE)
opt   = AdamW(probe.parameters(), lr=probe_lr, weight_decay=probe_wd)
ce    = nn.CrossEntropyLoss()

def _features(batch):
    ev, cen, kpm = gather_ctx_subset(batch)
    sub_kpm_patch = batch["pool_patch_kpm"].to(DEVICE)
    ctx_idx       = batch["ctx_patch_idx"].to(DEVICE)
    B, Pc         = ctx_idx.shape
    sub_kpm       = torch.gather(sub_kpm_patch, 1, ctx_idx)
    with torch.no_grad():
        g = enc(ev, cen, event_kpm=kpm)
    mask = (~sub_kpm).unsqueeze(-1).float()
    return (g * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

print(f"\ntraining probe for {n_probe_epochs} epochs...")
history_probe = {"train_loss": [], "train_acc": [], "test_acc": []}
for ep in range(1, n_probe_epochs + 1):
    probe.train()
    total_loss, total_correct, total = 0.0, 0, 0
    t0 = time.time()
    for batch in train_eval_loader:
        y = batch["label"].to(DEVICE)
        z = _features(batch)
        opt.zero_grad(set_to_none=True)
        logits = probe(z)
        loss = ce(logits, y)
        loss.backward()
        opt.step()
        total_loss    += loss.item() * y.size(0)
        total_correct += (logits.argmax(-1) == y).sum().item()
        total         += y.size(0)
    train_loss = total_loss / total
    train_acc  = total_correct / total

    probe.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            y = batch["label"].to(DEVICE)
            z = _features(batch)
            test_correct += (probe(z).argmax(-1) == y).sum().item()
            test_total   += y.size(0)
    test_acc = test_correct / test_total

    history_probe["train_loss"].append(train_loss)
    history_probe["train_acc"].append(train_acc)
    history_probe["test_acc"].append(test_acc)
    print(f"  ep {ep:02d}/{n_probe_epochs}  train_loss={train_loss:.4f}  "
          f"train_acc={train_acc:.4f}  test_acc={test_acc:.4f}  "
          f"({time.time()-t0:.1f}s)")

best_test  = max(history_probe["test_acc"])
final_test = history_probe["test_acc"][-1]
print(f"\nfinal probe accuracy:  test = {final_test:.4f}  best = {best_test:.4f}")

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ep_axis = range(1, n_probe_epochs + 1)
ax.plot(ep_axis, [a*100 for a in history_probe["train_acc"]], 'o-', label="train", color='C0')
ax.plot(ep_axis, [a*100 for a in history_probe["test_acc"]],  's-', label="test",  color='C3')
ax.set_xlabel("probe epoch"); ax.set_ylabel("accuracy (%)")
ax.set_title(f"final linear probe  (best test = {best_test*100:.2f}%)")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()


## 9. Training plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
if history["loss"]:
    axes[0].plot(range(1, len(history["loss"])+1), history["loss"], lw=1.5)
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("smooth_L1"); axes[0].set_title("JEPA loss")
    axes[0].grid(alpha=0.3)
if history["diag_log"]:
    steps = history["diag_steps"]
    axes[1].plot(steps, [d["pred_std"] for d in history["diag_log"]], label="pred_std", color="C0")
    axes[1].plot(steps, [d["tgt_std"]  for d in history["diag_log"]], label="tgt_std",  color="C2")
    axes[1].plot(steps, [d["cos"]      for d in history["diag_log"]], label="cos",      color="C3")
    axes[1].axhline(0.01, color='red', ls='--', alpha=0.4, label="collapse floor")
    axes[1].set_xlabel("step"); axes[1].set_title("training diagnostics"); axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)
if history.get("probe_accs"):
    eps, accs = zip(*history["probe_accs"])
    axes[2].plot(eps, [a*100 for a in accs], 'o-', color='C3')
    axes[2].set_ylim(0, 100); axes[2].set_xlabel("epoch"); axes[2].set_ylabel("probe acc (%)")
    axes[2].set_title(f"probe (best = {max(accs)*100:.2f}%)"); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()
